In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import urllib

# Python inclass practice: Basic statistics

## 1. First, let's revisit how to download USGS data!
### 1.1. Define site-specific info

In [ ]:
# First insert your USGS API Key here
API_KEY = 'YOUR_API_KEY_HERE'

args = {
    'monitoring_location_id': 'USGS-04216000',
    'parameter_code': '00060',              # discharge, cfs
    'statistic_id': '00003',                # daily mean
    'datetime': '1990-09-01/2023-08-31',
    'f': 'csv',
    'limit': 20000,
}

In [ ]:
query = urllib.parse.urlencode(args, safe='/')   # safe='/' keeps the date range intact

In [ ]:
query

## 1.2. Create the url and access the data using urllib

In [ ]:
niagara_url = f'https://api.waterdata.usgs.gov/ogcapi/v1/collections/daily/items?{query}'
print(niagara_url)

## 1.3. Read the data

In [ ]:
# With that we need to download the data and get it into pandas.
# To download the data we'll use the `urllib` module which is 
# built into the python "standard library" of stuff you get for
# free when you install python. We use the `urllib.request.urlopen`
# function which simply opens a connection to the url, just like 
# going to the url in your web browser. Then, we can put the `response`
# into `pd.read_table`. There are a lot of other parameters going 
# into this function now, and this is very common for when you scrape
# data directly from the internet because formats vary.

request = urllib.request.Request(niagara_url, headers={'X-Api-Key': API_KEY})
response = urllib.request.urlopen(request)

#  read the table
df = pd.read_csv(response)
df.head()

In [ ]:
# post processing data
# If you don't remember what this means, please go back and check lab3c_python_urllib.ipynb.
df = (
    df
    .rename(columns={'time': 'date',
                     'value': 'streamflow',
                     'approval_status': 'quality_flag'})
    .assign(date=lambda d: pd.to_datetime(d['date']))
    .set_index('date')
    .sort_index()
)

# only keep the columns we care about
df = df[['streamflow', 'quality_flag']]

df.head()

### 1.4. Practice #1: Please download following dataset
Site id: 04215000 (Cayuga Creek neear Lancaster NY) </br>
Start date: 1930-09-01 </br>
End date: 2023-08-31

In [ ]:
# INSERT your code here
# save data for Cayuga sites to **df1**

## 1.5. Concatenate multi-site information

First, we only want streamflow data,so when concatenating multi-site information, we only only keep the column "streamflow". However, if both sites use "streamflow" as their column names, how can we tell which is which? We also need to rename the column names to their USGS site ID.

In [ ]:
df_n = df[['streamflow']].rename({'streamflow':'04216000'},axis=1)
df_n

#### 1.5.0. Practice #2: Like we did to `df`, extract "streamflow" column from `df1`, and rename the column "streamflow" to its corresponding USGS site ID "04215000"

##### _n represents Niagara River and _c represents Cayuga Creek

In [ ]:
df_c = #INSERT your code here

### 1.5.1. concatenate two dataframes `df_n` and `df_c`

In [ ]:
df_concat = pd.concat([df_n,df_c],axis=1)

### 1.5.2. We can quickly take a look at the two time series by simply using `df.plot`!

In [ ]:
df_concat.plot()

### 1.5.3. Practice #3: We noticed that there is some missing data from the late 1960s to early 1970s. For simplicity, we will focus on the period when both sites have data. Let's select the data between `1980-01-01` and `2022-12-31`.
If you don't remember how to select time series, check Section **2.4.2** in https://github.com/act-hydro/GLY606_2024/blob/main/in_class_practice/python_practice/python_inclass_5_dataframe.ipynb

In [ ]:
df_concat_sel = #Insert your code here

### 1.5.4. Last sanity check: we do not want have any missing data in the dataset
`df.dropna()` will automatically drop the rows with missing values. </br>
If we want to drop the columns with missing values, we should use `df.dropna(axix=1)`

In [ ]:
df_concat_sel = df_concat_sel.dropna()

# 2. Statistics calcluations!

## 2.1. Mean values

In [ ]:
# just in case you did not get through the first section
# The data file "ython_inclass_practice_6.data.csv" is located in data folder
# please replace "#INSERT PATH HERE" using the correct path to file
df_concat_sel = pd.read_csv("#INSERT PATH HERE",index_col=0,parse_dates=True)

In [ ]:
df_concat_mean = df_concat_sel.mean()
df_concat_mean

### 2.1.1. How do we quickly visualize the mean values in a time series plot?

In [ ]:
fig,ax = plt.subplots(1,1,figsize=[5,3],dpi=300)
df_concat_sel[['04216000']].plot(ax=ax,lw=0.5)
ax.set_title('USGS streamflow @ Niagara River (Buffalo, NY)\n Site No 04216000')
ax.set_ylabel('streamflow [cfs]')
ax.axhline(df_concat_mean['04216000'],c='orange')

### 2.1.2. Practice #4: A quick recap of subplots!
Since Niagara River and Cayuga Creek near Lancaster NY have different flow volumns, it is preferrable to plot them separately. So let's use `plt.subplots` to show the time series and mean values for both sites. </br>

If you forgot how subplots works, check the bottom part of inclass practice 3 https://github.com/act-hydro/GLY606_2024/blob/main/in_class_practice/python_practice/python_inclass_3_intro_to_matplotlib.ipynb

In [ ]:
# Please insert your code below 

### 2.1.3. What we can learn from the time series plots above?
Drainage area for Niagara River in Buffalo NY: 263,700 square miles </br>
Drainage area for Cayuga Creek near Landcaster NY: 96.4 square miles </br>

Streamflow in smaller river basins like Cayuga Creek is usually more responsive to precipitation events, leading to a more spiky high flow events.


## 2.2. standard deviation (SD)
To calculate the standard deviation for a column in dataframe, we can simply use `df.std()`

In [ ]:
df_concat_sd = df_concat_sel.std(axis=0)

In [ ]:
df_concat_sd

## 2.3. Coefficient of variation
CV = (Standard Deviation) / (Mean) </br> 
A higher CV indicates greater relative variability in the data, meaning the data points are spread further apart compared to the mean

In [ ]:
df_concat_coeff_variation = df_concat_sd / df_concat_mean

In [ ]:
df_concat_coeff_variation

### Once again, Cayuga Creek has a much higher coefficient of variation than Niagara River, which echos our observations about the streamflow timeseries.

## 2.4. Box plots

We can use box plots to show the spread of the datasets!

In [ ]:
fig,axes = plt.subplots(1,2,figsize=[5,6],dpi=100)
axes[0].boxplot(df_concat_sel[['04216000']])
axes[0].set_ylabel("streamflow [cfs]")
axes[0].set_xticks([1],['0421600'])

axes[1].boxplot(df_concat_sel[['04215000']])
axes[1].set_xticks([1],['0421500'])
axes[1].set_ylim(0,)
# axes[1].set_yscale('log')
plt.tight_layout()


### 2.4.1. Practice #5: log-scale of y-axis
The box plot in the Cayuga River is largely influenced by the high flow events. What if we change the y-axis to log-scale? Please uncomment `ax.set_yscale('log')` and see what changes this line makes.

In [ ]:
fig,ax = plt.subplots(1,1,figsize=[6,10],dpi=300)
ax.boxplot(df_concat_sel[['04215000']],flierprops={'marker':'+','linewidth':0.2})
ax.set_xticks([1],['0421500'])
# ax.set_yscale('log')


# 3. PDF & CDF

## 3.1. Probability Density Function

For discrete sample data, we usually use histogram `plt.hist` to visualize it. </br>
Note: this plotting function has an option called `density`. If `density=False`, the plot will show the count falling within each bin. If `density=True`, the plot will show the probability density.

In [ ]:
fig,ax = plt.subplots(1, 1,figsize=[5,4],dpi=200)
ax.hist(df_concat_sel['04216000'],bins=np.arange(100000,350000,10000),density=True)
ax.set_title("PDF for streamflow in Niagara River")

### 3.1.1. It is important to choose bin size and scales wisely

For example, the streamflow regime in Cayuga River is very spiky. Most of the time, the flow is between 0-100 cfs, but it also had some very high flow days with thousands cfs. Therefore, if we use normal scale plots with fixed bins, it will look like left plot below. However, if we use log-scale for x-axis with smart selection of bin width, the plot looks much better (like the one on the right).

In [ ]:
fig,axes = plt.subplots(1, 2,figsize=[10,4],dpi=200)

axes[0].hist(df_concat_sel['04215000'],bins=np.arange(100,4000,100),
             density=True)
axes[1].hist(df_concat_sel['04215000'],bins=np.concatenate([[0,1,2,5,10,20,50,80],
                                                            np.arange(100,4000,100)]),
             density=True)
axes[1].set_xscale('log')
axes[0].set_title("Linear-scale")
axes[1].set_title("Log-scale")

fig.suptitle("PDF for streamflow in Cayuga Creek")

## 3.2. Cumulative Distribution Function

If we have a discrete array of samples, and we would like to know the CDF of the sample, then we can just sort the array. If we look at the sorted result, we'll realize that the smallest value represents 0% , and largest value represents 100 %. Then all other arrays are assigned with evenly distributed probabilities between 0% and 100%.

In [ ]:
# Calculate the cumulative proportion of the data that falls below each value
cumulative = np.linspace(0, 1, len(df_concat_sel['04216000']))

# Sort the data in ascending order
sorted_data = np.sort(df_concat_sel['04216000'])

# Calculate the cumulative proportion of the sorted data
cumulative_data = np.cumsum(sorted_data) / np.sum(sorted_data)

In [ ]:
# Plot the CDF
plt.figure(dpi=300)
plt.plot(sorted_data, cumulative_data, label='streamflow')
plt.ylim(0,1)

# Add labels and title
plt.xlabel("Streamflow (cfs)")
plt.ylabel("Cumulative Proportion")
plt.title("Cumulative Distribution Function (CDF) of Mean daily flow")
plt.legend()
plt.show()

## 3.2.1. Practice #6: plot CDF for Cayuga River. 

In [ ]:
# Please insert your code here